# PySpark Basics 

It walks through common PySpark patterns:
- Creating DataFrames in different ways
- Working with columns and rows
- Joining and aggregating data
- Reading data from files and APIs
- Saving results

You can run this notebook in a local PySpark environment **or** upload it to Databricks. Some cells use Databricks-specific helpers (like `display` and `dbutils`); if you're not on Databricks, use `df.show()` instead and skip those cells.

## 1. Set up a Spark session

If you're running on Databricks, a `SparkSession` named `spark` is already available. In a local environment, you can create one like this:

In [0]:
from pyspark.sql import SparkSession

# If you're on Databricks, you can skip this cell – Spark is already created.
spark = SparkSession.builder.appName("pyspark-basics-demo").getOrCreate()

spark

## 2. Import common data types and functions

We'll use the `pyspark.sql.types` and `pyspark.sql.functions` modules frequently.

In [0]:
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructType,
    StructField,
)
import pyspark.sql.types as T
import pyspark.sql.functions as F

print("Loaded pyspark.sql.types as T and pyspark.sql.functions as F")

## 3. Create DataFrames

### 3.1 From specified values

You can directly create a small DataFrame from Python data structures. This is useful for examples, tests, or small lookup tables.

In [0]:
# Create a simple DataFrame with schema inferred
df_children = spark.createDataFrame(
    data=[("Mikhail", 15), ("Zaky", 13), ("Zoya", 8)],
    schema=["name", "age"],
)

df_children.show()

In [0]:
# Create the same DataFrame, but with an explicit schema
schema_children = StructType([
    StructField("name", StringType(), nullable=True),
    StructField("age", IntegerType(), nullable=True),
])

df_children_with_schema = spark.createDataFrame(
    data=[("Mikhail", 15), ("Zaky", 13), ("Zoya", 8)],
    schema=schema_children,
)

df_children_with_schema.printSchema()
df_children_with_schema.show()

### 3.2 From a table (for Databricks / catalog-based setups)

If you're on Databricks with Unity Catalog available, you can create a DataFrame from a table path like
`<catalog>.<schema>.<table>`. The Databricks docs use the sample tables under `samples.tpch`.

If you're **not** on Databricks, you can skip this cell or replace the table name with one that exists in your environment.

In [0]:
# This requires a catalog table; works out-of-the-box on Databricks with the samples enabled.
table_name = "samples.tpch.customer"  # change this if needed

try:
    df_customer = spark.table(table_name)
    print(f"Loaded table: {table_name}")
    df_customer.show(5)
except Exception as e:
    print("Could not load table. If you're not on Databricks,",\
          "replace 'table_name' with a table that exists in your environment.")
    print("Error:", e)

### 3.3 From a CSV file

You can use `spark.read` with a given format (for example `csv`) to load data from files. Here we show a generic example. On Databricks, you could plug in a path from `/databricks-datasets` or from a Unity Catalog volume. Locally, just point to a CSV file on your filesystem.

In [0]:
# Set this to a real path before running, for example:
#   local:   data/customers.csv
#   dbx UC:  /Volumes/your_catalog/your_volume/your_file.csv
volume_file_path = ""  # TODO: fill in your path

if volume_file_path:
    df_csv = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(volume_file_path)
    )
    df_csv.show(5)
else:
    print("volume_file_path is empty; set it to a real CSV path before running this cell.")

### 3.4 From a JSON API response

You can also convert JSON data (for example from a REST API) into a Spark DataFrame. Here we use a small example with an open API. Adjust the URL to match your needs.

In [0]:
import requests

# Example: public JSON API (change to any API you want)
url = "https://api.fda.gov/drug/drugsfda.json?limit=20"
response = requests.get(url)

data_json = response.json()
print("Top-level keys in response:", list(data_json.keys()))

if "results" in data_json:
    # Remove 'products' and 'submissions' fields from each record to avoid schema inference error
    results_flat = [
        {k: v for k, v in rec.items() if k not in ["products", "submissions"]}
        for rec in data_json["results"]
    ]
    df_drugs = spark.createDataFrame(results_flat)
    df_drugs.printSchema()
    df_drugs.show(5)
    print("Note: 'products' and 'submissions' fields removed for schema inference. To handle nested arrays, define an explicit schema.")
else:
    print("Unexpected JSON structure; no 'results' key found.")

In [0]:
# Selecting nested JSON fields
if 'df_drugs' in globals():
    if "products" in df_drugs.columns:
        # Example: select the 'products' field (often an array) from the JSON
        df_products_only = df_drugs.select(df_drugs["products"])
        df_products_only.show(5, truncate=False)
    else:
        print("The 'products' column does not exist in df_drugs. It was removed in the previous cell to avoid schema inference errors.\n"
              "If you want to work with nested fields like 'products', you need to define an explicit schema when creating the DataFrame.")
else:
    print("df_drugs not defined – run the previous cell first.")

## 4. Transform data with DataFrames

Now let's look at common transformations: column operations, row operations, joins, and aggregations.

For these examples, we'll assume that `df_customer` exists. If you're not on Databricks, you can create a synthetic `df_customer` first.

In [0]:
# If df_customer is not available from a real table, create a small synthetic one for demos.
from pyspark.sql import Row

if 'df_customer' not in globals():
    sample_customers = [
        Row(c_custkey=1, c_acctbal=1000.50, c_mktsegment="AUTOMOBILE", c_nationkey=10, c_phone="111-111"),
        Row(c_custkey=2, c_acctbal=2500.00, c_mktsegment="BUILDING",   c_nationkey=20, c_phone="222-222"),
        Row(c_custkey=3, c_acctbal=800.00,  c_mktsegment="HOUSEHOLD",  c_nationkey=20, c_phone=""),
        Row(c_custkey=4, c_acctbal=None,    c_mktsegment="AUTOMOBILE", c_nationkey=30, c_phone=None),
    ]
    df_customer = spark.createDataFrame(sample_customers)
    print("Created synthetic df_customer for local demos.")

df_customer.show()

### 4.1 Column operations
**Select columns** using helper functions like `col` or string column names.

In [0]:
from pyspark.sql.functions import col, expr

# Using col()
df_selected = df_customer.select(
    col("c_custkey"),
    col("c_acctbal"),
)
df_selected.show()

# Using expr()
df_expr = df_customer.select(
    expr("c_custkey"),
    expr("c_acctbal"),
)
df_expr.show()

# Using string literals
df_str = df_customer.select("c_custkey", "c_acctbal")
df_str.show()

**Create and rename columns** with `withColumn` and `withColumnRenamed`, or use `alias` during aggregations.

In [0]:
# Create a new boolean column based on a condition
df_customer_flag = df_customer.withColumn(
    "balance_flag",
    col("c_acctbal") > 1000,
)

df_customer_flag.show()

In [0]:
# Rename a column
df_customer_flag_renamed = df_customer_flag.withColumnRenamed(
    "balance_flag", "balance_flag_renamed"
)

df_customer_flag_renamed.show()

In [0]:
# Cast a column type, for example from integer/number to string
df_casted = df_customer.withColumn("c_custkey", col("c_custkey").cast(StringType()))
df_casted.printSchema()
df_casted.show()

In [0]:
# Drop one or more columns
df_dropped = df_customer_flag_renamed.drop("balance_flag_renamed")
df_dropped.show()

df_dropped_multi = df_customer_flag_renamed.drop("c_phone", "balance_flag_renamed")
df_dropped_multi.show()

### 4.2 Row operations
**Filter rows**, remove duplicates, handle nulls, append and sort.

In [0]:
# Filter rows by a single condition
df_filtered_one = df_customer.filter(col("c_acctbal") > 1000)
df_filtered_one.show()

In [0]:
# Filter rows with multiple conditions (AND / OR)
df_filtered_two = df_customer.filter(
    (col("c_nationkey") == 20) & (col("c_acctbal") > 1000)
)
df_filtered_two.show()

df_filtered_or = df_customer.filter(
    (col("c_custkey") == 1) | (col("c_custkey") == 2)
)
df_filtered_or.show()

In [0]:
# Remove duplicate rows
df_unique = df_customer.distinct()
df_unique.show()

In [0]:
# Handle nulls: drop rows with any or all nulls in selected columns
df_no_nulls_any = df_customer.na.drop("any")
print("Drop rows with ANY nulls:")
df_no_nulls_any.show()

df_no_nulls_all_subset = df_customer.na.drop("all", subset=["c_acctbal", "c_phone"])
print("Drop rows where ALL selected columns are null:")
df_no_nulls_all_subset.show()

In [0]:
# Fill null or empty values
df_filled_acctbal = df_customer.na.fill(0, subset=["c_acctbal"])
print("Replace null balances with 0:")
df_filled_acctbal.show()

df_filled_phone = df_customer.na.replace([""], ["UNKNOWN"], subset=["c_phone"])
print("Replace empty phone strings with 'UNKNOWN':")
df_filled_phone.show()

In [0]:
# Append rows using union
df_appended_rows = df_filtered_or.union(df_filtered_one)
df_appended_rows.show()

In [0]:
# Sort rows by one or more columns
df_sorted = df_customer.orderBy(col("c_acctbal").desc(), col("c_custkey").asc())
df_sorted.show()

# Limit the number of rows after sorting
df_sorted.limit(3).show()

### 4.3 Join DataFrames

Joining two DataFrames is similar to joining tables in SQL. Let's create a small orders DataFrame and join it with `df_customer`.

In [0]:
from pyspark.sql import Row

orders_data = [
    Row(o_orderkey=100, o_custkey=1, o_totalprice=500.0),
    Row(o_orderkey=101, o_custkey=2, o_totalprice=1500.0),
    Row(o_orderkey=102, o_custkey=99, o_totalprice=999.0),  # no matching customer
]

df_order = spark.createDataFrame(orders_data)
df_order.show()

In [0]:
# Inner join: only rows where the join condition matches
df_join_inner = df_order.join(
    df_customer,
    on=(df_order["o_custkey"] == df_customer["c_custkey"]),
    how="inner",
)

df_join_inner.show()

In [0]:
# Left join: keep all rows from the left (orders), match customers where possible
df_join_left = df_order.join(
    df_customer,
    on=(df_order["o_custkey"] == df_customer["c_custkey"]),
    how="left",
)

df_join_left.show()

In [0]:
# Join with an additional condition
df_join_filtered = df_order.join(
    df_customer,
    on=((df_order["o_custkey"] == df_customer["c_custkey"]) &
        (df_order["o_totalprice"] > 1000)),
    how="inner",
)

df_join_filtered.show()

### 4.4 Aggregate data

Aggregations are similar to SQL `GROUP BY`. Use `.groupBy(...).agg(...)` with functions like `avg`, `sum`, `max`, and `min`.

In [0]:
from pyspark.sql.functions import avg, sum as sum_, max as max_, min as min_

# Average account balance by market segment
df_segment_balance = df_customer.groupBy("c_mktsegment").agg(
    avg("c_acctbal").alias("avg_account_balance"),
)

df_segment_balance.show()

In [0]:
# Group by multiple columns
df_segment_nation_balance = df_customer.groupBy("c_mktsegment", "c_nationkey").agg(
    avg("c_acctbal").alias("avg_account_balance"),
    sum_("c_acctbal").alias("sum_account_balance"),
)

df_segment_nation_balance.show()

## 5. Save DataFrames

You can save a DataFrame as a table (in catalog-aware systems like Databricks) or as files in different formats (for example Parquet, CSV).

In [0]:
# Example: write to Parquet files
output_path = "pyspark_basics_output.parquet"  # adjust path as needed

df_segment_balance.write.mode("overwrite").parquet(output_path)
print(f"Wrote aggregated results to {output_path}")

## 6. Next steps

- Explore more Spark SQL functions: https://spark.apache.org/docs/latest/api/python/
- Try loading your own datasets and applying the same patterns.
- If you're on Databricks, replace the synthetic data with tables from your own catalogs or with sample data under `samples` or `/databricks-datasets`.